# 02 — Quay Crane Allocation

**HYPAQ Ports · Initial educational reference · Apache-2.0**

Allocate two cranes across two vessels and two periods. A crane can serve at most
one vessel per period or remain idle. Each crane supplies four abstract work
units per active period; each vessel needs eight units.

Both vessels are assumed available in both periods. This example is independent
of the berth schedule in Exercise 01. Its objective is lexicographic: first
minimize unmet demand, then minimize active crane-slots. Oversupply is allowed
and served-demand metrics are clipped at demand.

**Implemented:** exact classical allocation and capacity checks. **Planned:** a
validated QUBO, Ket/KBW experiments and a timeline. This is a simplified allocation
exercise, not a complete QCSP with movement, non-crossing or interference.

For Colab, first follow `notebooks/README.md`. Run cells in order.

In [1]:
from pathlib import Path
import sys
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/hypaq_ports/reference.py").is_file()
             and (p / "data/synthetic").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the repository; see notebooks/README.md for Colab setup.")
sys.path.insert(0, str(ROOT / "src"))
from hypaq_ports.reference import (
    load_json, solve_berth_exact, berth_metrics, validate_berth_schedule,
    build_berth_qubo, qubo_energy, decode_berth_bits, qubo_to_ising,
    solve_crane_exact, evaluate_crane_schedule, apply_arrival_delay,
)
print("Repository:", ROOT.name)


Repository: hypaq-ports-starter


## 1. Solve and validate the tiny instance

Each of four crane-period pairs has three options: idle, V1 or V2. Exact enumeration visits all `3**4 = 81` assignments.

In [2]:
instance = load_json(ROOT / "data/synthetic/cranes_tiny_v1.json")
reference = solve_crane_exact(instance)
print(json.dumps(reference, indent=2))
metrics = evaluate_crane_schedule(instance, reference["schedule"])
assert metrics["feasible"]
assert metrics["total_unmet_units"] == 0
assert reference["assignments_examined"] == 81


{
  "status": "OPTIMAL",
  "method": "exhaustive_enumeration",
  "objective_order": [
    "total_unmet_units",
    "active_crane_slots"
  ],
  "schedule": [
    {
      "crane": "QC1",
      "slot": 0,
      "vessel": "V1"
    },
    {
      "crane": "QC1",
      "slot": 1,
      "vessel": "V1"
    },
    {
      "crane": "QC2",
      "slot": 0,
      "vessel": "V2"
    },
    {
      "crane": "QC2",
      "slot": 1,
      "vessel": "V2"
    }
  ],
  "metrics": {
    "feasible": true,
    "violations": [],
    "served_units": {
      "V1": 8,
      "V2": 8
    },
    "unmet_units": {
      "V1": 0,
      "V2": 0
    },
    "total_unmet_units": 0,
    "active_crane_slots": 4,
    "utilization": 1.0
  },
  "assignments_examined": 81
}


## 2. Increase demand beyond available capacity

Unmet demand is permitted and minimized. This is not the same as violating the crane exclusivity constraint. The reported optimal status refers to the best allocation for the stated objective, not a claim that all demand was met.

In [3]:
import copy
high_demand = copy.deepcopy(instance)
high_demand["vessels"][0]["demand_units"] = 100
stressed = solve_crane_exact(high_demand)
print(json.dumps(stressed["metrics"], indent=2))
assert stressed["metrics"]["feasible"]
assert stressed["metrics"]["total_unmet_units"] > 0


{
  "feasible": true,
  "violations": [],
  "served_units": {
    "V1": 16,
    "V2": 0
  },
  "unmet_units": {
    "V1": 84,
    "V2": 8
  },
  "total_unmet_units": 92,
  "active_crane_slots": 4,
  "utilization": 1.0
}


## 3. Reject an invalid allocation

The same crane cannot serve two vessels in the same period. Validate a deliberately invalid schedule.

In [4]:
invalid = [
    {"crane": "QC1", "vessel": "V1", "slot": 0},
    {"crane": "QC1", "vessel": "V2", "slot": 0},
]
check = evaluate_crane_schedule(instance, invalid)
print(json.dumps(check, indent=2))
assert not check["feasible"]


{
  "feasible": false,
  "violations": [
    "Crane assigned more than once in a period"
  ],
  "served_units": {
    "V1": 4,
    "V2": 4
  },
  "unmet_units": {
    "V1": 4,
    "V2": 4
  },
  "total_unmet_units": 8,
  "active_crane_slots": 2,
  "utilization": null
}


## Explore next

1. Change one crane's productivity and compare coverage and utilization.
2. Reduce demand and observe idle crane-periods under the secondary objective.
3. Propose a QUBO with explicit definitions for exclusivity and unmet-demand costs.
4. Validate its feasible optima before adding quantum execution.
5. Later, add vessel service windows to connect this model to berth allocation.

All productivity and demand values are illustrative abstract units. No terminal
capacity, savings or quantum advantage is inferred from this example.